# 02 — Market Sizing

**Purpose:** Quantify the overall Brazilian import market by chapter and NCM-4, identify concentration, and flag high-growth chapters relevant to QEntrega and Itatibense.

**Input:** `outputs/data/enriched.parquet`, `outputs/data/mart_chapter_month.parquet`, `outputs/data/mart_ncm4_country.parquet`

**Output:** Charts in `outputs/charts/`, chapter-level summary table.

**Scope:** All data unless explicitly scoped to complete periods. YoY comparisons are limited to complete periods (2025-only) vs. 2026 annualized estimate — clearly labeled.

**Core QEntrega/Itatibense chapters (pharma/chem/hazmat):** 28, 29, 30, 38, 39

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "outputs" / "data"
CHART_DIR    = PROJECT_ROOT / "outputs" / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (14, 5)})

# Chapters of primary interest for QEntrega/Itatibense
FOCUS_CHAPTERS = ["28", "29", "30", "38", "39"]
FOCUS_LABELS = {
    "28": "Ch28 Inorganic chem",
    "29": "Ch29 Organic chem",
    "30": "Ch30 Pharma",
    "38": "Ch38 Misc chem",
    "39": "Ch39 Plastics",
}

In [ ]:
enriched = pd.read_parquet(DATA_DIR / "enriched.parquet")
mart_cm   = pd.read_parquet(DATA_DIR / "mart_chapter_month.parquet")
mart_nc   = pd.read_parquet(DATA_DIR / "mart_ncm4_country.parquet")

print(f"enriched        : {len(enriched):,} rows")
print(f"mart_chapter_month: {len(mart_cm):,} rows")
print(f"mart_ncm4_country : {len(mart_nc):,} rows")

## 1. Total market snapshot — all years

In [ ]:
# Grand totals per year
annual = (
    enriched.groupby("CO_ANO")[["KG_LIQUIDO", "VL_FOB", "N_OPS"]]
    .agg({"KG_LIQUIDO": "sum", "VL_FOB": "sum", "N_OPS": "sum"})
    if "N_OPS" in enriched.columns
    else enriched.groupby("CO_ANO")[["KG_LIQUIDO", "VL_FOB"]].sum()
)

# Fallback: count rows if N_OPS column doesn't exist in enriched
if "N_OPS" not in enriched.columns:
    annual["N_OPS"] = enriched.groupby("CO_ANO").size()

annual["VL_FOB_bn"]  = (annual["VL_FOB"] / 1e9).round(2)
annual["KG_MM_mt"]   = (annual["KG_LIQUIDO"] / 1e9).round(2)
annual["N_OPS_k"]    = (annual["N_OPS"] / 1e3).round(1)

print("Annual market totals (FOB USD billions | million metric tons | k operations):")
display(annual[["VL_FOB_bn", "KG_MM_mt", "N_OPS_k"]].rename(
    columns={"VL_FOB_bn": "FOB USD bn", "KG_MM_mt": "MMt", "N_OPS_k": "Ops (k)"})
)
print("\nNote: 2026 = partial year (~Q1). Do not compare raw totals without annualizing.")

## 2. Top-20 chapters by FOB — 2025 full year

In [ ]:
# Use 2025 complete year for stable ranking
cm_2025 = mart_cm[mart_cm["CO_ANO"] == 2025]

chap_2025 = (
    cm_2025.groupby("CO_CAPITULO")[["KG_LIQUIDO", "VL_FOB", "N_OPS"]]
    .sum()
    .sort_values("VL_FOB", ascending=False)
    .reset_index()
)

chap_2025["VL_FOB_bn"]   = (chap_2025["VL_FOB"] / 1e9).round(2)
chap_2025["KG_MMt"]       = (chap_2025["KG_LIQUIDO"] / 1e6).round(1)  # thousand metric tons
chap_2025["share_pct"]    = (chap_2025["VL_FOB"] / chap_2025["VL_FOB"].sum() * 100).round(1)
chap_2025["is_focus"]     = chap_2025["CO_CAPITULO"].isin(FOCUS_CHAPTERS)

top20 = chap_2025.head(20)

print("Top-20 chapters by FOB USD (2025, full year):")
display(
    top20[["CO_CAPITULO", "VL_FOB_bn", "share_pct", "KG_MMt", "N_OPS"]]
    .rename(columns={"CO_CAPITULO": "Chapter", "VL_FOB_bn": "FOB USD bn",
                      "share_pct": "Share %", "KG_MMt": "kMt", "N_OPS": "Ops"})
    .reset_index(drop=True)
)

# Focus chapters in top-20?
focus_in_top20 = top20[top20["is_focus"]]["CO_CAPITULO"].tolist()
missing_focus  = [c for c in FOCUS_CHAPTERS if c not in focus_in_top20]
print(f"\nFocus chapters in top-20: {focus_in_top20}")
if missing_focus:
    print(f"⚠️  Focus chapters NOT in top-20: {missing_focus}")
else:
    print("✓ All 5 focus chapters present in top-20.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = ["#e04b3a" if c in FOCUS_CHAPTERS else "#4a90d9" for c in top20["CO_CAPITULO"]]
bars = ax.barh(top20["CO_CAPITULO"][::-1], top20["VL_FOB_bn"][::-1], color=colors[::-1])

ax.set_xlabel("FOB USD (billions)")
ax.set_title("Top-20 HS Chapters by Import FOB Value — Brazil 2025 (full year)", pad=12)

# Label bars
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{w:.1f}B", va="center", fontsize=8)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#e04b3a", label="Focus chapters (28/29/30/38/39)"),
    Patch(color="#4a90d9", label="Other chapters"),
], loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(CHART_DIR / "market_top20_chapters.png", bbox_inches="tight")
plt.show()
print("Saved: market_top20_chapters.png")

## 3. Focus chapters deep-dive — absolute size and share

In [ ]:
focus_summary = (
    chap_2025[chap_2025["is_focus"]]
    .sort_values("VL_FOB", ascending=False)
    .reset_index(drop=True)
)

# Add rank within all chapters
rank_map = {r["CO_CAPITULO"]: i+1 for i, r in chap_2025.iterrows()}
focus_summary["rank_all"] = focus_summary["CO_CAPITULO"].map(rank_map)
focus_summary["label"] = focus_summary["CO_CAPITULO"].map(FOCUS_LABELS)

print("Focus chapter summary — 2025 full year:")
display(
    focus_summary[["CO_CAPITULO", "label", "rank_all", "VL_FOB_bn", "share_pct", "N_OPS"]]
    .rename(columns={
        "CO_CAPITULO": "Chapter", "label": "Description",
        "rank_all": "Rank", "VL_FOB_bn": "FOB USD bn",
        "share_pct": "Share %", "N_OPS": "Ops",
    })
)

total_focus_fob = focus_summary["VL_FOB"].sum()
total_fob = chap_2025["VL_FOB"].sum()
print(f"\nCombined FOB USD (focus chapters): ${total_focus_fob/1e9:.1f}B "
      f"({100*total_focus_fob/total_fob:.1f}% of total market)")

## 4. Origin concentration — HHI and CR4/CR10 by chapter

In [ ]:
# HHI and concentration ratios per chapter using mart_ncm4_country
# Aggregate by chapter (CO_CAPITULO) + country
nc_with_chap = mart_nc.copy()
nc_with_chap["CO_CAPITULO"] = nc_with_chap["CO_POSICAO"].str[:2]

chap_country = (
    nc_with_chap.groupby(["CO_CAPITULO", "CO_PAIS", "NO_PAIS_ING"])["VL_FOB"]
    .sum()
    .reset_index()
)

def concentration_metrics(group: pd.DataFrame) -> pd.Series:
    """HHI (0–10000) and CR4/CR10 for a group already sorted by VL_FOB desc."""
    sorted_grp = group.sort_values("VL_FOB", ascending=False)
    total = sorted_grp["VL_FOB"].sum()
    if total == 0:
        return pd.Series({"HHI": np.nan, "CR4": np.nan, "CR10": np.nan, "n_origins": 0})
    shares = sorted_grp["VL_FOB"] / total
    hhi = round((shares ** 2).sum() * 10000, 0)
    cr4  = round(shares.head(4).sum() * 100, 1)
    cr10 = round(shares.head(10).sum() * 100, 1)
    return pd.Series({"HHI": hhi, "CR4": cr4, "CR10": cr10, "n_origins": len(sorted_grp)})

conc = (
    chap_country.groupby("CO_CAPITULO")
    .apply(concentration_metrics, include_groups=False)
    .reset_index()
)

# Merge with FOB totals for context
conc = conc.merge(
    chap_2025[["CO_CAPITULO", "VL_FOB_bn", "share_pct"]],
    on="CO_CAPITULO", how="left"
).sort_values("VL_FOB_bn", ascending=False)

conc["HHI_label"] = pd.cut(
    conc["HHI"],
    bins=[0, 1500, 2500, 10001],
    labels=["Competitive (<1500)", "Moderate (1500–2500)", "Concentrated (>2500)"],
)

print("Concentration metrics by chapter (top-30 by FOB, 2025):")
display(
    conc.head(30)[["CO_CAPITULO", "VL_FOB_bn", "n_origins", "CR4", "CR10", "HHI", "HHI_label"]]
    .rename(columns={"CO_CAPITULO": "Chapter", "VL_FOB_bn": "FOB bn",
                      "n_origins": "# Origins", "HHI_label": "Competition"})
    .reset_index(drop=True)
)

In [ ]:
# Scatter: FOB scale vs. HHI — bubble = n_origins
focus_conc = conc[conc["CO_CAPITULO"].isin(FOCUS_CHAPTERS)]
other_conc = conc[~conc["CO_CAPITULO"].isin(FOCUS_CHAPTERS)].head(25)

fig, ax = plt.subplots(figsize=(12, 6))

sc_other = ax.scatter(
    other_conc["VL_FOB_bn"], other_conc["HHI"],
    s=other_conc["n_origins"] * 3, alpha=0.5, color="#4a90d9", label="Other chapters"
)
sc_focus = ax.scatter(
    focus_conc["VL_FOB_bn"], focus_conc["HHI"],
    s=focus_conc["n_origins"] * 3, alpha=0.85, color="#e04b3a", zorder=5, label="Focus chapters"
)

for _, row in focus_conc.iterrows():
    ax.annotate(
        FOCUS_LABELS.get(row["CO_CAPITULO"], row["CO_CAPITULO"]),
        (row["VL_FOB_bn"], row["HHI"]),
        textcoords="offset points", xytext=(8, 4), fontsize=8,
    )

ax.axhline(1500, color="gray", linestyle="--", linewidth=0.8, label="HHI=1500 (competitive threshold)")
ax.axhline(2500, color="orange", linestyle="--", linewidth=0.8, label="HHI=2500 (concentrated threshold)")
ax.set_xlabel("FOB USD (billions) — 2025")
ax.set_ylabel("HHI (origin concentration)")
ax.set_title("Market size vs. origin concentration by HS Chapter — bubble = # of source countries")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(CHART_DIR / "market_hhi_scatter.png", bbox_inches="tight")
plt.show()
print("Saved: market_hhi_scatter.png")

In [ ]:
# Top-5 origins per focus chapter
for chap in FOCUS_CHAPTERS:
    grp = (
        chap_country[chap_country["CO_CAPITULO"] == chap]
        .sort_values("VL_FOB", ascending=False)
        .head(5)
        .reset_index(drop=True)
    )
    if grp.empty:
        continue
    total = grp["VL_FOB"].sum()
    grp["share_pct"] = (grp["VL_FOB"] / chap_country[chap_country["CO_CAPITULO"] == chap]["VL_FOB"].sum() * 100).round(1)
    grp["VL_FOB_mm"] = (grp["VL_FOB"] / 1e6).round(1)
    print(f"\n{'='*55}")
    print(f"Chapter {chap} — {FOCUS_LABELS[chap]} — top-5 origins:")
    display(grp[["CO_PAIS", "NO_PAIS_ING", "VL_FOB_mm", "share_pct"]]
            .rename(columns={"CO_PAIS": "Code", "NO_PAIS_ING": "Country",
                              "VL_FOB_mm": "FOB USD MM", "share_pct": "Share %"}))

## 5. YoY comparison — 2025 full year vs. 2026 annualized estimate

In [ ]:
# Determine how many months of 2026 data are present
months_2026 = mart_cm[mart_cm["CO_ANO"] == 2026]["CO_MES"].nunique()
print(f"Months of 2026 data available: {months_2026}")

if months_2026 == 0:
    print("No 2026 data — skipping YoY comparison.")
else:
    # Annualization factor
    annualize = 12 / months_2026
    print(f"Annualization factor: 12 / {months_2026} = {annualize:.2f}x")
    print("⚠️  Annualized 2026 is an ESTIMATE. Seasonal effects not adjusted.")

In [ ]:
if months_2026 > 0:
    cm_2026 = mart_cm[mart_cm["CO_ANO"] == 2026]

    chap_2026_raw = cm_2026.groupby("CO_CAPITULO")["VL_FOB"].sum().reset_index(name="VL_FOB_2026_raw")
    chap_2026_raw["VL_FOB_2026_ann"] = chap_2026_raw["VL_FOB_2026_raw"] * annualize

    yoy = chap_2025[["CO_CAPITULO", "VL_FOB"]].rename(columns={"VL_FOB": "VL_FOB_2025"})
    yoy = yoy.merge(chap_2026_raw[["CO_CAPITULO", "VL_FOB_2026_ann"]], on="CO_CAPITULO", how="left")
    yoy["YoY_pct"] = ((yoy["VL_FOB_2026_ann"] / yoy["VL_FOB_2025"]) - 1) * 100
    yoy = yoy.sort_values("VL_FOB_2025", ascending=False)

    yoy["FOB_2025_bn"] = (yoy["VL_FOB_2025"] / 1e9).round(2)
    yoy["FOB_2026_ann_bn"] = (yoy["VL_FOB_2026_ann"] / 1e9).round(2)
    yoy["is_focus"] = yoy["CO_CAPITULO"].isin(FOCUS_CHAPTERS)

    print("YoY growth — top-20 chapters by 2025 FOB (2026 annualized estimate):")
    display(
        yoy.head(20)[["CO_CAPITULO", "FOB_2025_bn", "FOB_2026_ann_bn", "YoY_pct"]]
        .rename(columns={"CO_CAPITULO": "Chapter", "FOB_2025_bn": "2025 FOB bn",
                          "FOB_2026_ann_bn": "2026 Ann. bn", "YoY_pct": "YoY %"})
        .round({"YoY %": 1})
        .reset_index(drop=True)
    )
    print(f"\n⚠️  2026 figures are annualized from {months_2026} months. "
          "Treat as directional, not precise.")

In [ ]:
if months_2026 > 0 and len(yoy) > 0:
    # YoY bar chart for focus chapters
    focus_yoy = yoy[yoy["is_focus"]].sort_values("YoY_pct", ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: FOB absolute comparison (2025 vs. 2026 annualized)
    ax = axes[0]
    x = np.arange(len(focus_yoy))
    w = 0.35
    ax.bar(x - w/2, focus_yoy["FOB_2025_bn"], w, label="2025", color="#4a90d9")
    ax.bar(x + w/2, focus_yoy["FOB_2026_ann_bn"], w, label="2026 (ann.)",
           color="#e04b3a", alpha=0.85, hatch="//")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [FOCUS_LABELS.get(c, c) for c in focus_yoy["CO_CAPITULO"]],
        rotation=15, ha="right", fontsize=8
    )
    ax.set_ylabel("FOB USD (billions)")
    ax.set_title("Focus chapters: 2025 vs. 2026 ann.")
    ax.legend(fontsize=8)

    # Right: YoY % for top-20 chapters
    ax2 = axes[1]
    yoy_top20 = yoy.head(20).sort_values("YoY_pct")
    colors_yoy = ["#e04b3a" if c in FOCUS_CHAPTERS else
                  ("#27ae60" if v >= 0 else "#c0392b")
                  for c, v in zip(yoy_top20["CO_CAPITULO"], yoy_top20["YoY_pct"])]
    ax2.barh(yoy_top20["CO_CAPITULO"], yoy_top20["YoY_pct"], color=colors_yoy)
    ax2.axvline(0, color="black", linewidth=0.8)
    ax2.set_xlabel("YoY % (2026 ann. vs 2025)")
    ax2.set_title("YoY growth — top-20 chapters")

    plt.suptitle(f"⚠️ 2026 annualized from {months_2026} months — directional only",
                 fontsize=9, color="darkorange", y=1.01)
    plt.tight_layout()
    plt.savefig(CHART_DIR / "market_yoy_chapters.png", bbox_inches="tight")
    plt.show()
    print("Saved: market_yoy_chapters.png")

## 6. Monthly trend — focus chapters (2025)

In [ ]:
trend = (
    mart_cm[
        (mart_cm["CO_ANO"] == 2025) &
        (mart_cm["CO_CAPITULO"].isin(FOCUS_CHAPTERS))
    ]
    .sort_values(["CO_CAPITULO", "CO_MES"])
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (metric, label, scale) in enumerate([
    ("VL_FOB", "FOB USD (millions)", 1e6),
    ("N_OPS",  "Operations",          1),
]):
    ax = axes[ax_idx]
    for chap, grp in trend.groupby("CO_CAPITULO"):
        ax.plot(grp["CO_MES"], grp[metric] / scale,
                marker="o", markersize=4,
                label=FOCUS_LABELS.get(chap, chap))
    ax.set_xlabel("Month (2025)")
    ax.set_ylabel(label)
    ax.set_title(f"{label} — focus chapters 2025")
    ax.legend(fontsize=7)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig(CHART_DIR / "market_focus_monthly_trend.png", bbox_inches="tight")
plt.show()
print("Saved: market_focus_monthly_trend.png")

## 7. Top-20 NCM-4 headings by FOB — 2025

In [ ]:
# Aggregate mart_ncm4_country to NCM-4 level (sum across all countries)
ncm4_total = (
    mart_nc.groupby("CO_POSICAO")[["KG_LIQUIDO", "VL_FOB", "N_OPS"]]
    .sum()
    .reset_index()
    .sort_values("VL_FOB", ascending=False)
)

# Note: mart_ncm4_country uses all available years. For a fair snapshot filter
# use enriched directly for 2025 only.
ncm4_2025 = (
    enriched[enriched["CO_ANO"] == 2025]
    .groupby("CO_POSICAO")[["KG_LIQUIDO", "VL_FOB"]]
    .sum()
    .reset_index()
    .sort_values("VL_FOB", ascending=False)
)
ncm4_2025["VL_FOB_mm"] = (ncm4_2025["VL_FOB"] / 1e6).round(1)
ncm4_2025["share_pct"] = (ncm4_2025["VL_FOB"] / ncm4_2025["VL_FOB"].sum() * 100).round(2)
ncm4_2025["is_focus"]  = ncm4_2025["CO_POSICAO"].str[:2].isin(FOCUS_CHAPTERS)

# Attach description if available
if "NO_NCM_POR" in enriched.columns:
    desc = (
        enriched[["CO_POSICAO", "NO_NCM_POR"]]
        .dropna(subset=["NO_NCM_POR"])
        .drop_duplicates("CO_POSICAO")
    )
    ncm4_2025 = ncm4_2025.merge(desc, on="CO_POSICAO", how="left")
    display_cols = ["CO_POSICAO", "NO_NCM_POR", "VL_FOB_mm", "share_pct"]
else:
    display_cols = ["CO_POSICAO", "VL_FOB_mm", "share_pct"]

top20_ncm4 = ncm4_2025.head(20).reset_index(drop=True)
print("Top-20 NCM-4 headings by FOB USD (2025):")
display(top20_ncm4[display_cols])

In [ ]:
# Top-20 NCM-4 within focus chapters
focus_ncm4 = (
    ncm4_2025[ncm4_2025["is_focus"]]
    .head(20)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(14, 6))
colors = ["#e04b3a" if c in FOCUS_CHAPTERS else "#4a90d9"
          for c in top20_ncm4["CO_POSICAO"].str[:2]]
ax.barh(top20_ncm4["CO_POSICAO"][::-1], top20_ncm4["VL_FOB_mm"][::-1],
        color=colors[::-1])
ax.set_xlabel("FOB USD (millions)")
ax.set_title("Top-20 NCM-4 headings by FOB — Brazil 2025 (red = focus chapters)")
plt.tight_layout()
plt.savefig(CHART_DIR / "market_top20_ncm4.png", bbox_inches="tight")
plt.show()
print("Saved: market_top20_ncm4.png")

## 8. Port of entry (URF) — where does the volume land?

In [ ]:
urf_summary = (
    enriched[enriched["CO_ANO"] == 2025]
    .groupby(["CO_URF", "NO_URF"])["VL_FOB"]
    .sum()
    .reset_index()
    .sort_values("VL_FOB", ascending=False)
    .head(15)
)
urf_summary["VL_FOB_bn"]  = (urf_summary["VL_FOB"] / 1e9).round(2)
urf_summary["share_pct"]  = (urf_summary["VL_FOB"] /
                               enriched[enriched["CO_ANO"] == 2025]["VL_FOB"].sum() * 100).round(1)
urf_summary["label"] = urf_summary["NO_URF"].fillna(urf_summary["CO_URF"])

print("Top-15 ports of entry by FOB USD (2025):")
display(urf_summary[["CO_URF", "label", "VL_FOB_bn", "share_pct"]]
        .rename(columns={"CO_URF": "URF Code", "label": "Port/URF",
                          "VL_FOB_bn": "FOB bn", "share_pct": "Share %"})
        .reset_index(drop=True))

In [ ]:
# Focus chapters: which URFs handle them?
focus_urf = (
    enriched[
        (enriched["CO_ANO"] == 2025) &
        (enriched["CO_CAPITULO"].isin(FOCUS_CHAPTERS))
    ]
    .groupby(["CO_URF", "NO_URF", "CO_CAPITULO"])["VL_FOB"]
    .sum()
    .reset_index()
)

# Pivot: URF × chapter heatmap-ready table
top_urfs = (
    focus_urf.groupby("CO_URF")["VL_FOB"].sum()
    .sort_values(ascending=False)
    .head(12).index
)

pivot_urf = (
    focus_urf[focus_urf["CO_URF"].isin(top_urfs)]
    .pivot_table(index="CO_URF", columns="CO_CAPITULO", values="VL_FOB",
                 aggfunc="sum", fill_value=0)
    / 1e6
).round(1)

# Replace URF codes with names where available
urf_name_map = (
    focus_urf[["CO_URF", "NO_URF"]].dropna(subset=["NO_URF"])
    .drop_duplicates("CO_URF")
    .set_index("CO_URF")["NO_URF"]
    .to_dict()
)
pivot_urf.index = [urf_name_map.get(c, c) for c in pivot_urf.index]
pivot_urf.columns.name = "Chapter"

print("FOB USD (millions) by port × focus chapter — 2025:")
display(pivot_urf.sort_values(pivot_urf.columns.tolist(), ascending=False))

## 9. "So what?" — Sales intelligence summary

In [ ]:
print("=" * 65)
print("  MARKET SIZING — SALES INTELLIGENCE SUMMARY")
print("=" * 65)

# 1. Focus chapter combined weight
total_fob_all = chap_2025["VL_FOB"].sum()
focus_fob = chap_2025[chap_2025["is_focus"]]["VL_FOB"].sum()
print(f"\nFocus chapters (28/29/30/38/39) combined 2025 FOB:")
print(f"  ${focus_fob/1e9:.1f}B — {100*focus_fob/total_fob_all:.1f}% of total market")

# 2. Growth signal
if months_2026 > 0:
    focus_yoy_df = yoy[yoy["is_focus"]].copy()
    growing = focus_yoy_df[focus_yoy_df["YoY_pct"] > 0].sort_values("YoY_pct", ascending=False)
    if len(growing):
        print(f"\nGrowing focus chapters (2026 annualized vs 2025):")
        for _, r in growing.iterrows():
            print(f"  Ch{r['CO_CAPITULO']} {FOCUS_LABELS.get(r['CO_CAPITULO'],'')}: "
                  f"+{r['YoY_pct']:.1f}%")

# 3. Port concentration
top3_urf = urf_summary.head(3)
top3_share = top3_urf["share_pct"].sum()
top3_names = " / ".join(top3_urf["label"].tolist())
print(f"\nTop-3 ports handle {top3_share:.0f}% of total FOB:")
print(f"  {top3_names}")

print("\n" + "-" * 65)
print("QEntrega (air freight — GRU/VCP):")
print("  → Ch30 (Pharma) and Ch29 (Org. chem): time-sensitive, air-eligible.")
print("  → High unit value = freight cost is a small % of FOB = low price resistance.")
print("  → Target: importers in SP/MG from US/EU origin at GRU/VCP.")
print()
print("Itatibense (road/sea — Santos, Itajaí, Paranaguá):")
print("  → Ch39 (Plastics) and Ch28/38 (industrial chem): high volume, sea dominant.")
print("  → Destination states: SP, SC, PR, MG.")
print("  → Target: bulk chemical and plastic importers clearing through Santos/Itajaí.")
print()
print("Data gap: no importer names in public Comex Stat.")
print("To unlock company-level prospecting: add Logcomex or ImportGenius DI data.")